# Πού γεννιέται η βλάβη: δύο κελιά

**Τι ξέρουμε.** Το ανταγωνιστικό frame μέσα στα μηνύματα ρίχνει τη συνεργασία.
Το φίλτρο F1 έδειξε ότι αυτό **δεν χρειάζεται παράδοση**: με 90–100% των
μηνυμάτων μπλοκαρισμένα, όσοι δεν έλαβαν τίποτα συνεργάστηκαν στο 0,03–0,32,
δηλαδή όσο και όταν παραδίδονταν όλα, ενώ το `silence` δίνει 0,30–0,70.

**Τι δεν ξέρουμε.** Δύο πράγματα:

1. Το F1 μπλοκάρει *σχεδόν* τα πάντα, και το λεξιλόγιό του επικαλύπτεται με τις
   λέξεις της ίδιας της οδηγίας. Θέλουμε την καθαρή εκδοχή: **τίποτα δεν
   παραδίδεται**, απέναντι στο `silence`, όπου δεν γράφεται καν μήνυμα.
2. Αν η βλάβη φτάνει στους **παραλήπτες**. Ένας πράκτορας με την οδηγία
   ανάμεσα σε τρεις ουδέτερους το απαντά: αν πέσουν και οι τρεις, η βλάβη
   ταξιδεύει· αν πέσει μόνο ο γράφων, είναι αυτο-επίδραση.

| κελί | ποιος γράφει ανταγωνιστικά | τι παραδίδεται | τι κρίνει |
|---|---|---|---|
| **(α)** `block_all` | και οι 4 | **τίποτα** | χρειάζεται η βλάβη παραλήπτη; |
| **(β)** `agents0` | μόνο ο κόμβος-κέντρο | όλα | μεταδίδεται στους αναγνώστες; |

Και τα δύο στον **αστέρα**, PD, n=5, ώστε να συγκρίνονται με τα υπάρχοντα
κελιά του αστέρα: `framing_competitive` (όλοι γράφουν, όλα παραδίδονται),
`silence` (κανείς δεν γράφει) και `no_comm` (χωρίς κανάλι).

**Στον αστέρα ο γράφων του (β) είναι ο πράκτορας 0, δηλαδή ο κόμβος-κέντρο**:
είναι ο μόνος που τον ακούν και τα τρία φύλλα, οπότε η μετάδοση έχει την
καλύτερη δυνατή ευκαιρία να φανεί.

## Τι δεν αλλάζει

Ίδιο παιχνίδι και αποδόσεις, 16 γύροι, κρυφός ορίζοντας, μνήμη 10, T=0,7,
μηνύματα ως 20 λέξεις, ίδιο `max_tokens` ανά μοντέλο με τα υπόλοιπα κελιά του
session B. Αλλάζουν μόνο δύο πράγματα, και τα δύο προαιρετικά:

- `--message-filter block_all`: το κανάλι δεν παραδίδει τίποτα. Ο αποστολέας
  βλέπει κανονικά το δικό του μήνυμα, όπως σε κάθε φιλτραρισμένο run.
- `--policy-agents 0`: μόνο ο πράκτορας 0 παίρνει την ανταγωνιστική οδηγία,
  οι άλλοι τρεις γράφουν συνηθισμένα μηνύματα.

Τα runs πάνε σε δικούς τους φακέλους (`..._block_all`, `..._agents0`) και τα
κελιά τους έχουν δικές τους ετικέτες (`+blockall`, `+agents0`), οπότε δεν
μπορούν να μπερδευτούν με το `framing_competitive` του πλέγματος.

## Setup

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN` (για Llama και τα δύο gemma)

Μετά **Save & Run All**, αλλάζοντας μόνο το `STEP`.

| βήμα | μοντέλο | runs | ~ώρες |
|---|---|---|---|
| 1 | gemma-2-2b | 10 | 2,4 |
| 2 | Qwen2.5-7B | 10 | 2,0 |
| 3 | Llama-3.1-8B | 10 | 2,8 |
| 4 | Qwen3-4B | 10 | 3,2 |
| 5 | gemma-2-9b | 10 | 4,4 |

**Σύνολο 50 runs, ~15 ώρες.** Κάθε βήμα τρέχει και τα δύο κελιά του μοντέλου
στην ίδια συνεδρία, ώστε να συγκρίνονται χωρίς διαφορά συνεδρίας.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

assert os.path.exists('campaign.py'), f'campaign.py δεν είναι στο {os.getcwd()}'
for f in ('campaign.py', 'run_all_scenarios.py'):
    assert '--topology-aware-comm-prompt' in open(f, encoding='utf-8').read(), (
        f'Το {f} δεν έχει --topology-aware-comm-prompt -- κάνε git push πρώτα.')

print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())


## Ο έλεγχος που κρίνει το πείραμα

Τρία πράγματα, χωρίς GPU: ότι το `block_all` δεν παραδίδει τίποτα, ότι με
`policy_agents=[0]` μόνο ο πράκτορας 0 παίρνει την ανταγωνιστική οδηγία, και
ότι το prompt των υπολοίπων είναι ταυτόσημο με του κανονικού cheap talk.

In [ ]:
import sys
sys.path.insert(0, '.')
from config import ExperimentConfig, ModelConfig
from engine import make_engine
from message_policies import apply_filter, get_extra_message_instruction

delivered, blocked = apply_filter('block_all', 'we will dominate this round')
assert delivered == '' and blocked, (delivered, blocked)
print('block_all: παραδίδει', repr(delivered), '| blocked =', blocked)

cfg = ExperimentConfig(game='pd', condition='cheap_talk', topology='star',
                       message_policy='framing', framing_type='competitive',
                       policy_agents=[0], model=ModelConfig(provider='local'))
agents = make_engine(cfg).build_agents(client=None)
for a in agents:
    instr = get_extra_message_instruction(a.message_policy, a.framing_type)
    tag = 'ΑΝΤΑΓΩΝΙΣΤΙΚΗ ΟΔΗΓΙΑ' if 'competitive tone' in instr else 'συνηθισμένο μήνυμα'
    print(f'  πράκτορας {a.agent_id} ({"κέντρο" if a.agent_id == 0 else "φύλλο"}): {tag}')
assert 'competitive tone' in get_extra_message_instruction(agents[0].message_policy,
                                                           agents[0].framing_type)
assert all(get_extra_message_instruction(a.message_policy, a.framing_type) == ''
           for a in agents[1:]), 'τα φύλλα δεν πρέπει να πάρουν οδηγία -- σταμάτα'

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    print('Χωρίς HF_TOKEN (εντάξει μόνο για Qwen):', e)


## Ποιο βήμα

In [ ]:
STEP = 1          # <-- ΜΟΝΟ ΑΥΤΟ ΑΛΛΑΖΕΙΣ (1-5)

MODELS = ['google/gemma-2-2b-it', 'Qwen/Qwen2.5-7B-Instruct',
          'meta-llama/Llama-3.1-8B-Instruct', 'Qwen/Qwen3-4B',
          'google/gemma-2-9b-it']
MODEL = MODELS[STEP - 1]
NEEDS_HF = MODEL.startswith(('meta-llama/', 'google/'))

COMMON = ['--session', 'B', '--topology', 'star',
          '--scenarios', 'framing_competitive', '--games', 'pd']
CALLS = [('block_all: κανένα μήνυμα δεν παραδίδεται',
          [*COMMON, '--message-filter', 'block_all']),
         ('agents0: μόνο ο κόμβος-κέντρο γράφει ανταγωνιστικά',
          [*COMMON, '--policy-agents', '0'])]
EXPECTED = 5

def args_for(extra):
    return ['--model', MODEL, *extra]

print(f'βήμα {STEP}: {MODEL}')
for name, extra in CALLS:
    print(f'   {name}  -> {EXPECTED} runs')

## Preview — κοστίζει μηδέν

Σε κάθε κλήση πρέπει να δεις `topology : star`, `expecting : 5 run files`, και
τη γραμμή `FILTER` ή `WRITERS` με δικό της φάκελο.

In [ ]:
import os, subprocess, sys

ok = True
if NEEDS_HF and not os.environ.get('HUGGINGFACE_API_KEY'):
    ok = False
    print('ΛΕΙΠΕΙ ΤΟ HF_TOKEN -- το μοντέλο είναι gated, θα αποτύχει σε ~1 λεπτό.')
for name, extra in CALLS:
    print('=' * 70)
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(extra), '--dry-run'],
                       capture_output=True, text=True)
    out = r.stdout.strip()
    print(out[:900])
    if r.returncode != 0:
        ok = False
        print('ΣΦΑΛΜΑ:', (r.stderr or r.stdout).strip()[-300:])
    needles = ['topology     : star', f'expecting    : {EXPECTED} run files',
               'block_all' if 'block_all' in extra else 'WRITERS']
    for needle in needles:
        if needle not in out:
            ok = False
            print(f'ΛΕΙΠΕΙ ΑΠΟ ΤΟ ΠΛΑΝΟ: {needle!r}')

assert ok, 'Κάποιο plan απέτυχε -- μη συνεχίσεις.'
print('\n' + '=' * 70)
print('όλα τα πλάνα εντάξει')

## Εκτέλεση

In [ ]:
import subprocess, sys, time

t0 = time.time()
results = []
for name, extra in CALLS:
    print('\n' + '=' * 70)
    print(f'{MODEL}  {name}   ({(time.time() - t0) / 3600:.1f} h μέχρι τώρα)')
    print('=' * 70, flush=True)
    s = time.time()
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(extra)])
    mins = (time.time() - s) / 60
    status = 'OK' if r.returncode == 0 else f'FAILED (exit {r.returncode})'
    results.append((name, status, mins))
    print(f'\n--> {name}: {status}, {mins:.0f} λεπτά', flush=True)

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for name, s, mins in results:
    print(f'  {s:22s} {mins:6.0f} λ   {name}')
print(f'\nσύνολο {(time.time() - t0) / 3600:.1f} h')

## Έλεγχος των δεδομένων και πρώτη ματιά

Για το `block_all` επιβεβαιώνει ότι κανένα μήνυμα δεν παραδόθηκε ενώ όλα
γράφτηκαν. Για το `agents0` χωρίζει τη συνεργασία του **γράφοντος** (κόμβος
0) από των **αναγνωστών** (τα τρία φύλλα): εκεί κρίνεται η μετάδοση.

In [ ]:
import glob, json, os, sys, statistics
sys.path.insert(0, '.')
from analysis import summarise_run

print(f"{'κελί':30s} {'n':>3s} {'coop':>6s} {'γράφων':>8s} {'αναγνώστες':>12s} {'blocked':>9s}")
for d in sorted(glob.glob('/kaggle/working/results/*')):
    tag = os.path.basename(d)
    if not (tag.endswith('_block_all') or tag.endswith('_agents0')):
        continue
    coop, hub, leaf, blocked, n = [], [], [], [], 0
    for p in glob.glob(f'{d}/**/*.json', recursive=True):
        if os.path.basename(p).startswith('_') or f'{os.sep}zips{os.sep}' in p:
            continue
        rec = json.load(open(p, encoding='utf-8'))
        s = summarise_run(rec)
        n += 1
        coop.append(s['coop_rate_overall'])
        hub.append(s['coop_rate_hub'])
        leaf.append(s['coop_rate_leaf'])
        for x in rec['history']:
            b = x.get('messages_blocked', {})
            if b:
                blocked.append(sum(b.values()) / len(b))
        if tag.endswith('_block_all'):
            for x in rec['history']:
                assert all(v == '' for v in x['messages'].values()), f'παραδόθηκε μήνυμα! {p}'
                assert any(v for v in x.get('messages_composed', {}).values()), \
                    f'δεν γράφτηκε μήνυμα -- το κελί δεν δοκιμάζει τίποτα: {p}'
    if n:
        b = f'{statistics.fmean(blocked):9.0%}' if blocked else f'{"--":>9s}'
        print(f'{tag:30s} {n:3d} {statistics.fmean(coop):6.3f} '
              f'{statistics.fmean(hub):8.3f} {statistics.fmean(leaf):12.3f} {b}')

print('\nαναφορά, αστέρας PD (πλέγμα):')
print('  framing_competitive  0.01-0.36     silence  0.04-0.70     no_comm  0.04-0.67')

## Μάζεμα

In [ ]:
import glob, os, shutil

zips = sorted(glob.glob('/kaggle/working/*_block_all*.zip')
              + glob.glob('/kaggle/working/*_agents0*.zip'))
print(f'{len(zips)} zip:')
for z in zips:
    print(f'   {os.path.getsize(z)/1e6:6.1f} MB  {os.path.basename(z)}')
if zips:
    box = '/kaggle/working/harm_source'
    os.makedirs(box, exist_ok=True)
    for z in zips:
        shutil.copy(z, box)
    out = shutil.make_archive(
        f"/kaggle/working/harm_source_step{STEP}_{MODEL.split('/')[-1]}", 'zip', box)
    print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')
else:
    print('\nΚΑΝΕΝΑ ZIP -- δες τον απολογισμό παραπάνω.')

## Πώς διαβάζεται

Τρεις συγκρίσεις, όλες στον ίδιο αστέρα και στο ίδιο μοντέλο:

1. **`block_all` έναντι `framing_competitive`.** Αν είναι ίδια, η παράδοση δεν
   παίζει κανένα ρόλο: αρκεί να γράψει κανείς το μήνυμα.
2. **`block_all` έναντι `silence`.** Και στα δύο κανείς δεν λαμβάνει τίποτα.
   Η μόνη διαφορά είναι ότι στο `block_all` ο πράκτορας **έγραψε** ανταγωνιστικό
   μήνυμα. Ό,τι απόσταση υπάρχει εδώ, τη δημιουργεί η συγγραφή.
3. **`agents0`, γράφων έναντι αναγνωστών.** Αν πέφτουν και οι τρεις αναγνώστες,
   η βλάβη μεταδίδεται. Αν πέφτει μόνο ο κόμβος 0, είναι αυστηρά αυτο-επίδραση,
   και τότε το «φιλτράρισμα του καναλιού» ήταν εξαρχής λάθος μοχλός.

Κατέβασε το zip στο `diplomatikh/harm_source/`, δικός του φάκελος.

**Μία σημείωση για το gemma-2-9b:** το session B τρέχει με `max_tokens` 192,
όσο και τα runs του RQ4 (F1, F3, επαναλήψεις), ενώ το `framing_competitive`
του πλέγματος στον αστέρα είχε 160. Η σύγκριση με τα κελιά του RQ4 είναι
καθαρή· η σύγκριση με το κελί του πλέγματος κουβαλά αυτή τη διαφορά.